# include_StJude_mutation_burden.ipynb

Likely pathogenic variants for SJC + PBTA dataset, MW U test and boxplots.  
See also likely_pathogenic_mutation_burden.ipynb.

In [ ]:
import cyvcf2
import pandas as pd
import numpy as np
import os

In [ ]:
def parse_bcftools_stats(file):
    # read a bcftools stats output, and
    # return the total number of variants
    with open(file) as f:
        for line in f:
            if line.startswith('SN	0	number of records:'):
                count=line.rstrip().split('\t')[-1]
                count=int(count)
    try:
        return count
    except:
        print("could not read file:",file)
        raise

In [ ]:
def get_variant_counts(path):
    # for all .stats.txt files in path,
    # create a dictionary of the format:
    # return {sample : count}
    d = {}
    for filename in os.listdir(path):
        full_path = os.path.join(path, filename)
        if full_path.endswith('.stats.txt'):
            count = parse_bcftools_stats(full_path)
            #print(count)
            filename = filename.split('.')[0]
            d[filename] = count
    return d

def get_variant_count_df():
    # return a dataframe of the format:
    # file_id, all_counts, pathogenic_counts

    # get all counts
    all_counts = get_variant_counts('./data/stats/all')
    df1 = pd.Series(all_counts)
    df1.name = "all_counts"
    df = pd.DataFrame(df1)
    # get likely pathogenic counts
    pathogenic_counts = get_variant_counts('./data/stats/likely_pathogenic')
    df2 = pd.Series(pathogenic_counts)
    df2.name = "pathogenic_counts"
    # check the indices are the same
    assert set(df1.index) == set(df2.index)
    # merge and return
    df = df.join(df2)
    return df

In [ ]:
def read_manifest(file='./data/pedpancan/manifest_20250910_143948.csv'):
    df = pd.read_csv(file)
    df=df[df.name.str.endswith('.gz')]
    df['index'] = df.name.map(lambda x: os.path.basename(x).split('.')[0])
    df = df.set_index('index')
    df = df[['Kids First Biospecimen ID']].copy()
    return df

In [ ]:
def read_biosample_table(file='./data/pedpancan/Supplementary Tables.xlsx'):
    df = pd.read_excel(file,sheet_name='2. Biosamples',index_col='biosample_id')
    df = df[df.in_unique_patient_set].copy()
    return df

def merge_counts(manifest,counts):
    assert set(manifest.index) == set(counts.index)
    df = manifest.join(counts)
    return df

def check_file_bs_ids(example):
    # check no duplicates
    assert len(example.index) == len(example.index.unique())
    assert len(example['Kids First Biospecimen ID']) == len(example['Kids First Biospecimen ID'].unique())
    # check no missing values
    assert sum(example.index.isna()) == 0
    assert sum(example['Kids First Biospecimen ID'].isna()) == 0
    # check 1:1 mapping
    assert len(example.index) == len(example['Kids First Biospecimen ID'])
    return

def merge_ecDNA_counts(ecDNA_df,counts_df):
    check_file_bs_ids(counts_df)
    counts_df['file_id'] = counts_df.index
    counts_df = counts_df.set_index('Kids First Biospecimen ID')
    df = ecDNA_df.join(counts_df,how='inner')
    return df

In [ ]:
df = merge_ecDNA_counts(read_biosample_table(),
                   merge_counts(read_manifest(),get_variant_count_df())
                  )
df.head()

In [ ]:
df = df.loc[:,['cancer_type','amplicon_class','all_counts','pathogenic_counts']]
df

In [ ]:
def get_sj_variant_count_df():
    # return a dataframe of the format:
    # file_id, all_counts, pathogenic_counts

    # get all counts
    all_counts = get_variant_counts('./data/sj_stats/all')
    df1 = pd.Series(all_counts)
    df1.name = "all_counts"
    df = pd.DataFrame(df1)
    # get likely pathogenic counts
    pathogenic_counts = get_variant_counts('./data/sj_stats/likely_pathogenic')
    df2 = pd.Series(pathogenic_counts)
    df2.name = "pathogenic_counts"
    # check the indices are the same
    assert set(df1.index) == set(df2.index)
    # merge and return
    df = df.join(df2)
    df.index = df.index.str.rsplit("_", n=1).str[0]
    return df

In [ ]:
get_sj_variant_count_df()

In [ ]:
def read_sj_manifest(file='./data/pedpancan/somatic_vcfs.tsv'):
    df = pd.read_table(file)
    df = df[df['vcf_file_name'] != 'SJST030131_D3_G1.Somatic.vcf.gz']
    df.index = df["vcf_file_name"].str.rsplit("_", n=1).str[0]
    return df

def merge__sj_ecDNA_counts(ecDNA_df,counts_df):
    df = ecDNA_df.join(counts_df,how='inner')
    return df

In [ ]:
sj_df = merge__sj_ecDNA_counts(read_biosample_table(),
                               merge_counts(read_sj_manifest(),get_sj_variant_count_df()))
sj_df.head()

In [ ]:
sj_df = sj_df.loc[:,['cancer_type','amplicon_class','all_counts','pathogenic_counts', ]]
sj_df

In [ ]:
df_all = pd.concat([df, sj_df], axis=0, ignore_index=False)
df_all['batch'] = df_all.index.astype(str).str[:2].map({'BS': 'openPBTA', 'SJ': 'St.Jude'})
df_all

In [ ]:
df_all['amplicon_class'].value_counts()

## Test

In [ ]:
import scipy
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compare_somatic_mutation_burden_osc(df, test=scipy.stats.mannwhitneyu):
    df['log_counts'] = np.log1p(df.all_counts)
    sns.boxplot(
        data = df,
        x = 'amplicon_class',
        #y = 'all_counts'
        y = 'log_counts',
        order = ['ecDNA','no amplification','intrachromosomal']
    )

    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['log_counts']
            b = df[df.amplicon_class == classes[j]]['log_counts']
            print(test(a,b))
    return

def compare_pathogenic_somatic_mutation_burden(df, test=scipy.stats.mannwhitneyu):
    
    sns.violinplot(
        data = df,
        x = 'amplicon_class',
        y = 'pathogenic_counts',
        density_norm = 'area',
        order = ['ecDNA','no amplification','intrachromosomal']
    )
    sns.despine()
    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['pathogenic_counts']
            b = df[df.amplicon_class == classes[j]]['pathogenic_counts']
            print(test(a,b))
    return

def boxplot_compare_pathogenic_somatic_mutation_burden(df, test=scipy.stats.mannwhitneyu):
    df['log_counts'] = np.log1p(df.pathogenic_counts)
    sns.boxplot(
        data = df,
        x = 'amplicon_class',
        y = 'log_counts',
        order = ['ecDNA','no amplification','intrachromosomal']
    )

    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['log_counts']
            b = df[df.amplicon_class == classes[j]]['log_counts']
            print(test(a,b))
    return


In [ ]:
compare_somatic_mutation_burden_osc(df_all)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/compare_somatic_mutation_burden.svg')
plt.show()

In [ ]:
boxplot_compare_pathogenic_somatic_mutation_burden(df_all)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/compare_pathogenic_somatic_mutation_burden.svg')
plt.show()

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
def stratify_ecDNA_by_tumor_type(df):
    df = df.loc[:,['amplicon_class','cancer_type']]
    df = df.groupby(['cancer_type','amplicon_class']).size()
    return df

In [ ]:
stratify_ecDNA_by_tumor_type(df_all)

In [ ]:
df_MBL = df_all[df_all['cancer_type'].isin(['MBL'])].copy()
df_HGG = df_all[df_all['cancer_type'].isin(['HGG'])].copy()

In [ ]:
compare_somatic_mutation_burden_osc(df_MBL)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/MBL_somatic_mutation_burden.svg')
plt.show()

In [ ]:
boxplot_compare_pathogenic_somatic_mutation_burden(df_MBL)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/MBL_pathogenic_somatic_mutation_burden.svg')
plt.show()

In [ ]:
compare_somatic_mutation_burden_osc(df_HGG)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/HGG_somatic_mutation_burden.svg')
plt.show()

In [ ]:
boxplot_compare_pathogenic_somatic_mutation_burden(df_HGG)
plt.ylabel('mutations')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.savefig('figure/HGG_pathogenic_somatic_mutation_burden.svg')
plt.show()